# Music genre classification
The goal of this project is to extract the necessary features from the raw audio files, and use them to train the kNN, ANN and CNN models with it. After that, it could be compared to models that were trained with the dataset provided CSV files. Only 30 v 30 or also v 3?
- extracted audio features manually ✔     
- train kNN on extracted features   ✔       
    - train kNN on provided features ✔     
    - train kNN on provided features (3s)
- train ANN on extracted features
    - train ANN on provided features
    - train ANN on provided features (3s)
- train CNN on provided spectrograms

### Importing GTZAN dataset

In [151]:
import os # to interact with OS: file/directory opeartions, envoronmental variables...
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"        
os.environ["MKL_NUM_THREADS"] = "1"        
os.environ["NUMEXPR_NUM_THREADS"] = "1"   # these 4 were added to prevent the python kernel from dying when
import pandas as pd # just, used to import the csvs. dataframe manipulation (csv/excel)

In [152]:
# Path to Dataset
PATH = "/home/ket/Documents/gtzan/archive/Data/genres_original"
PATH_CSV_30 = "/home/ket/Documents/gtzan/archive/Data/features_30_sec.csv"
PATH_CSV_3 = "/home/ket/Documents/gtzan/archive/Data/features_3_sec.csv"
MFCC_N = 14 # How may MFCC to keep (from 20)

In [153]:
# Import CSV and look at it
df_30 = pd.read_csv(PATH_CSV_30)
df_3 = pd.read_csv(PATH_CSV_3)
df_30.head(3)
df_30.tail(3)
# df_30.describe
#df_30.columns # 60 COLUMNS
# pandas quickstart: https://github.com/techwithtim/PanadasTutorial

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
997,rock.00097.wav,661794,0.432142,0.075268,0.081651,0.000322,2077.526598,231657.968040,1927.293153,74717.124394,...,33.597008,-12.845291,36.367264,3.440978,36.001110,-12.588070,42.502201,-2.106337,29.865515,rock
998,rock.00098.wav,661794,0.362485,0.091506,0.083860,0.001211,1398.699344,240318.731073,1818.450280,109090.207161,...,46.324894,-4.416050,43.583942,1.556207,34.331261,-5.041897,47.227180,-3.590644,41.299088,rock
999,rock.00099.wav,661794,0.358401,0.085884,0.054454,0.000336,1609.795082,422203.216152,1797.213044,120115.632927,...,59.167755,-7.069775,73.760391,0.028346,76.504326,-2.025783,72.189316,1.155239,49.662510,rock


In [154]:
# Pregled podatkov, any null values? 
# df_30.isnull().sum() # no null values
# df_30.info()

It appears that no data is missing
# Data preprocessing
The data set was imported, now it has to be determined which of the contained data, or rather clomuns, are relevant. There's 60 columns, let's take a look.
### What is needed
`filename` is not needed, can be removed, `label` has the genre . These three are needed because of the project description I wrote: MFCCs, ZCR, and Chroma features. But I see there's other features I will require like Spectral Centroids, Tempo and RMS (Root Mean Square).
Also do I need both avg and means? yea
### What is useful
`length`, `chroma_stft_mean`, `chroma_stft_var`, `rms_mean`
`rms_var`, `spectral_centroid_mean`, `spectral_centroid_var`,
`spectral_bandwidth_mean`, `spectral_bandwidth_var`, `rolloff_mean`,
`rolloff_var`, `zero_crossing_rate_mean`, `zero_crossing_rate_var`,
`harmony_mean`, `harmony_var`, `perceptr_mean`, `perceptr_var`, `tempo`
### What is not nedeed
`filename` and probably we could do without all the MFCCs. I remember 13 being enough.

In [155]:
# This code is reused below
# MFCC_N = 20
# col = df_30.pop("label")
# df_30.insert(0, "label", col)
# df_30.drop("filename", inplace=True, axis=1)
# # this loop removes as many MFCCs as defined at the top
# for n in range(20-MFCC_N):
#     name1 = f"mfcc{20-n}_var"
#     name2 = f"mfcc{20-n}_mean"
#     # print(name1, " and ", name2)
#     df_30.drop(name1, inplace=True, axis=1)
#     df_30.drop(name2, inplace=True, axis=1)
# df_30.columns



## Split Train & Test kNN - 30s

In [156]:
# ARCHIVED
# # https://www.geeksforgeeks.org/machine-learning/how-to-split-a-dataset-into-train-and-test-sets-using-python/
# # https://pylearnai.com/machine-learning/csv-to-prediction-machine-learning-python/
# # https://gao-hongnan.github.io/gaohn-galaxy/machine_learning/neighbours/k_nearest_neighbours/knn_feature_scaling.html 25.6.26
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.neighbors import KNeighborsClassifier

# ### variables
# MFCC_N = 20
# K_N = 30
# TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
# RANDOM_STATE = 100
# ###

# #omega loop to collect all the possible variations
# for i in range(MFCC_N):
#     for ii in range(K_N):
#         for iii in TEST_SIZE:
#             for iiii in range(RANDOM_STATE):

#                 col = df_30.pop("label")
#                 df_30.insert(0, "label", col)
#                 df_30.drop("filename", inplace=True, axis=1)
#                 # this loop removes as many MFCCs as defined at the top
#                 for n in range(20-MFCC_N):
#                     name1 = f"mfcc{20-n}_var"
#                     name2 = f"mfcc{20-n}_mean"
#                     # print(name1, " and ", name2)
#                     df_30.drop(name1, inplace=True, axis=1)
#                     df_30.drop(name2, inplace=True, axis=1)

#                 X = df_30.drop(columns=["label"])
#                 y = df_30['label'] # by genre
#                 # NORMALIZATION - we need this because "The K-NN algorithm relies on the Euclidean
#                 # distance between data points, making it highly sensitive to the scale of 
#                 # the features. If there’s a discrepancy in the scale across different features,
#                 # the feature with the larger scale will overshadow the others, leading to 
#                 # biased predictions."
#                 scale = StandardScaler()
#                 X_scaled = scale.fit_transform(X)

#                 X_train, X_test, y_train, y_test = train_test_split(
#                     X_scaled, y,
#                     test_size=iii, # 70 train 30 test
#                     random_state=iiii, # idk, tutorial had 42
#                     stratify=y # such that 30 songs from each genre are used for testing
#                     )

#                 # KNN
#                 knn = KNeighborsClassifier(n_neighbors=K_N).fit(X_train, y_train) # For MFCC = 20, K = 10 gives the best accuracy : 0.6767
#                 y_test_normalized_preds = knn.predict(X_test)                    # For MFCC = 13, K = 10 gives the best accuracy : 0.7134
#                 print("KNN accuracy:", knn.score(X_test, y_test))                # For MFCC = 14, K = 10 gives the best accuracy : 0.68
#                 print()


#                 # writing it down so I dont need to do it manually
#                 CSV_PATH_KNN_30 = "knn-data-df-30.csv"
#                 features = {
#                     "MFCC_N": [i],
#                     "TEST_SIZE": [iii],
#                     "RANDOM_STATE": [iiii],
#                     "K_N": [ii],
#                     "ACCURACY": [knn.score(X_test, y_test)]
#                 }

#                 if os.path.exists(CSV_PATH_KNN_30):
#                     df = pd.read_csv(CSV_PATH_KNN_30)
#                     df = pd.concat([df, pd.DataFrame(features)], ignore_index=True)
#                 else:
#                     df = pd.DataFrame(features)
#                 df.to_csv(CSV_PATH_KNN_30, index=False)

#                 print(df)


In [157]:
# COMMENTED TO NOT RUN EVERY TIME
# # https://www.geeksforgeeks.org/machine-learning/how-to-split-a-dataset-into-train-and-test-sets-using-python/
# # https://pylearnai.com/machine-learning/csv-to-prediction-machine-learning-python/
# # https://gao-hongnan.github.io/gaohn-galaxy/machine_learning/neighbours/k_nearest_neighbours/knn_feature_scaling.html 25.6.26
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.neighbors import KNeighborsClassifier
# from tqdm import tqdm

# ### variables
# MFCC_N = 20
# K_N = 30
# TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
# RANDOM_STATE = [0, 42, 99]
# CSV_PATH_KNN_30 = "knn-data-df-30.csv"
# ###


# #omega loop to collect all the possible variations
# results = []

# df_base = df_30.copy()
# col = df_base.pop("label")
# df_base.insert(0, "label", col)
# df_base.drop("filename", inplace=True, axis=1)

# for i in tqdm(range(1, MFCC_N+1), desc="kNN Data collection progress"):       
#     print("We're at MFCC: ", MFCC_N)
#     df_work = df_base.copy()

#     for n in range(20 - i):
#         df_work.drop(f"mfcc{20-n}_var", inplace=True, axis=1)
#         df_work.drop(f"mfcc{20-n}_mean", inplace=True, axis=1)

#     X = df_work.drop(columns=["label"])
#     y = df_work["label"]
#     X_scaled = StandardScaler().fit_transform(X)

#     for k in tqdm(range(1, K_N+1), desc="K-Neighbors", leave=False):      
#         for test_size in tqdm(TEST_SIZE, desc="Test size", leave=False):
#             for seed in tqdm(RANDOM_STATE, desc="seed", leave=False): 
                
#                 # col = df_work.pop("label")
#                 # df_work.insert(0, "label", col)
#                 # df_work.drop("filename", inplace=True, axis=1)        

#                 X_train, X_test, y_train, y_test = train_test_split(
#                     X_scaled, y, test_size=test_size, random_state=seed, stratify=y
#                 )

#                 knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
#                 acc = knn.score(X_test, y_test)

#                 results.append({"MFCC_N": i, "K_N": k, "TEST_SIZE": test_size,
#                                  "RANDOM_STATE": seed, "ACCURACY": acc})


# pd.DataFrame(results).to_csv(CSV_PATH_KNN_30, index=False)


After brief manual testing, the K that gave the best results was `K=10`, specifically with `MFCC_N=13`.
To the end of finding a combination of parametets that yield a higher accuracy, I automated some data collection - will be used later.
These were the parameters and their range of values:
`MFCC_N = [1-20]`
`K_N = [1-30]`
`TEST_SIZE = [0.1, 0.2, 0.3, 0.4]`
`RANDOM_STATE = [0, 42, 99]`


In [158]:
df_knn = pd.read_csv("/home/ket/Desktop/Jupyter ML/knn-data-df-30.csv")
df_knn_sorted = df_knn.sort_values(by="ACCURACY", ascending=False)

df_knn_sorted.head(10)

,MFCC_N,K_N,TEST_SIZE,RANDOM_STATE,ACCURACY
6840,20,1,0.1,0,0.770
4717,14,4,0.1,42,0.770
4729,14,5,0.1,42,0.760
5092,15,5,0.2,42,0.755
4381,13,6,0.1,42,0.750
6864,20,3,0.1,0,0.750
3973,12,2,0.1,42,0.750
6206,18,8,0.1,99,0.750
4741,14,6,0.1,42,0.750
5089,15,5,0.1,42,0.750


## Feature extraction
In the project description I wrote that I would extract the needed features manually. To achieve that I will use the `AudioFeaturizer` library. The out put data is not 1:1 as the data provided by the dataset but it should do.
Crucial to note that one of the original audio files seems to be corrupted as it is simply not working, and there is nothing that can be extracted from it. The file in question is `jazz.00054.wav`.

In [159]:
# https://pypi.org/project/AudioFeaturizer/
from AudioFeaturizer.audio_featurizer import *
# just testing 
test =audio_process("/home/ket/Documents/gtzan/archive/Data/genres_original/jazz/jazz.00055.wav")
print(type(test))
# genres = os.listdir(PATH)
# print("Genres: ", genres)
results = []

# COMMENTED SO IT DOESNT EXTRACT FEATURES EVERY TIME
for genre in tqdm(genres, desc="Feature exctraction"):
    genre_path = os.path.join(PATH, genre)
    for filename in os.listdir(genre_path):
        if(filename != "jazz.00054.wav"): # broken
            file_path = os.path.join(genre_path, filename)

            try:
                features = audio_process(file_path)
                row = features.iloc[0].to_dict()
                row["genre"] = genre
                row["filename"] = filename
                results.append(row)
            except Exception as e:
                print("[Error]: Something went wrong")

extr_df = pd.DataFrame(results)
extr_df.to_csv("extracted-features.csv", index=False)


<class 'pandas.core.frame.DataFrame'>


Feature exctraction:   0%|          | 0/10 [00:20<?, ?it/s]


KeyboardInterrupt: 

The audio features have now been extracted. As for the spectrograms, according to the project description the ones provided by the dataset will be used.
## Split Train & Test kNN - (extracted)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from tqdm import tqdm

### variables
MFCC_N = 20
K_N = 30
TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
RANDOM_STATE = [0, 42, 99]
CSV_PATH_KNN_EXTR = "knn-extracted-features.csv"
###


#omega loop to collect all the possible variations
results = []

df_base = extr_df.copy()
col = df_base.pop("genre")
df_base.insert(0, "genre", col)
df_base.drop("filename", inplace=True, axis=1)
#

for i in tqdm(range(1, MFCC_N+1), desc="kNN Data collection progress"):       
    print("We're at MFCC: ", MFCC_N)
    df_work = df_base.copy()

    for n in range(20 - i):
        df_work.drop(f"mfcc{20-n}", inplace=True, axis=1)

    X = df_work.drop(columns=["genre"])
    y = df_work["genre"]
    X_scaled = StandardScaler().fit_transform(X)

    for k in tqdm(range(1, K_N+1), desc="K-Neighbors", leave=False):      
        for test_size in tqdm(TEST_SIZE, desc="Test size", leave=False):
            for seed in tqdm(RANDOM_STATE, desc="seed", leave=False): 
                
                # col = df_work.pop("label")
                # df_work.insert(0, "label", col)
                # df_work.drop("filename", inplace=True, axis=1)        

                X_train, X_test, y_train, y_test = train_test_split(
                    X_scaled, y, test_size=test_size, random_state=seed, stratify=y
                )

                knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
                acc = knn.score(X_test, y_test)

                results.append({"MFCC_N": i, "K_N": k, "TEST_SIZE": test_size,
                                 "RANDOM_STATE": seed, "ACCURACY": acc})


pd.DataFrame(results).to_csv(CSV_PATH_KNN_EXTR, index=False)


kNN Data collection progress:   0%|          | 0/20 [00:00<?, ?it/s]

We're at MFCC:  20






























































































































































































































































































































































































































































































































































































kNN Data collection progress:   5%|▌         | 1/20 [00:04<01:31,  4.83s/it]

We're at MFCC:  20




























































































































































































































































































































































































































































































































































































kNN Data collection progress:  10%|█         | 2/20 [00:08<01:18,  4.37s/it]

...










































































































































































































































































































































































































































































































































































kNN Data collection progress:  90%|█████████ | 18/20 [01:05<00:06,  3.18s/it]

We're at MFCC:  20










































































































































































































































































































































































































































































































































































kNN Data collection progress:  95%|█████████▌| 19/20 [01:08<00:03,  3.17s/it]

We're at MFCC:  20




















































































































































































































































































































































































































































































































































































kNN Data collection progress: 100%|██████████| 20/20 [01:12<00:00,  3.62s/it]


In [ ]:
df_knn_ext = pd.read_csv("/home/ket/Desktop/Jupyter ML/knn-extracted-features.csv")
df_knn_ext_sorted = df_knn.sort_values(by="ACCURACY", ascending=False)

df_knn_ext_sorted.head(10)

,MFCC_N,K_N,TEST_SIZE,RANDOM_STATE,ACCURACY
272,1,23,0.3,99,0.510000
708,2,30,0.1,0,0.510000
704,2,29,0.3,99,0.506667
332,1,28,0.3,99,0.506667
344,1,29,0.3,99,0.506667
1028,3,26,0.3,99,0.506667
716,2,30,0.3,99,0.503333
1040,3,27,0.3,99,0.503333
680,2,27,0.3,99,0.503333
356,1,30,0.3,99,0.503333


It would appear that the missing mean/avg columns in the extracted features does not impact accuracy at all, in fact the results are identical.
## Split Train & Test kNN - 3s

In [ ]:
# ~22 minutes
# ### variables
# MFCC_N = 20
# K_N = 30
# TEST_SIZE = [0.1, 0.2, 0.3, 0.4]
# RANDOM_STATE = [0, 42, 99]
# CSV_PATH_KNN_3 = "knn-data-df-3.csv"
# # ###


# # #omega loop to collect all the possible variations
# results = []

# df_base = df_3.copy()
# col = df_base.pop("label")
# df_base.insert(0, "label", col)
# df_base.drop("filename", inplace=True, axis=1)

# for i in tqdm(range(1, MFCC_N+1), desc="kNN Data collection progress"):       
#     print("We're at MFCC: ", MFCC_N)
#     df_work = df_base.copy()

#     for n in range(20 - i):
#         df_work.drop(f"mfcc{20-n}_var", inplace=True, axis=1)
#         df_work.drop(f"mfcc{20-n}_mean", inplace=True, axis=1)

#     X = df_work.drop(columns=["label"])
#     y = df_work["label"]
#     X_scaled = StandardScaler().fit_transform(X)

#     for k in tqdm(range(1, K_N+1), desc="K-Neighbors", leave=False):      
#         for test_size in tqdm(TEST_SIZE, desc="Test size", leave=False):
#             for seed in tqdm(RANDOM_STATE, desc="seed", leave=False): 
                
#                 # col = df_work.pop("label")
#                 # df_work.insert(0, "label", col)
#                 # df_work.drop("filename", inplace=True, axis=1)        

#                 X_train, X_test, y_train, y_test = train_test_split(
#                     X_scaled, y, test_size=test_size, random_state=seed, stratify=y
#                 )

#                 knn = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
#                 acc = knn.score(X_test, y_test)

#                 results.append({"MFCC_N": i, "K_N": k, "TEST_SIZE": test_size,
#                                  "RANDOM_STATE": seed, "ACCURACY": acc})


# pd.DataFrame(results).to_csv(CSV_PATH_KNN_3, index=False)

kNN Data collection progress:   0%|          | 0/20 [00:00<?, ?it/s]

We're at MFCC:  20


We're at MFCC:  20


We're at MFCC:  20


We're at MFCC:  20


...

We're at MFCC:  20


We're at MFCC:  20


We're at MFCC:  20


We're at MFCC:  20


We're at MFCC:  20


In [ ]:
df_knn_3 = pd.read_csv("/home/ket/Desktop/Jupyter ML/knn-data-df-3.csv")
df_knn_3_sorted = df_knn.sort_values(by="ACCURACY", ascending=False)

df_knn_3_sorted.head(10)

,MFCC_N,K_N,TEST_SIZE,RANDOM_STATE,ACCURACY
272,1,23,0.3,99,0.510000
708,2,30,0.1,0,0.510000
704,2,29,0.3,99,0.506667
332,1,28,0.3,99,0.506667
344,1,29,0.3,99,0.506667
1028,3,26,0.3,99,0.506667
716,2,30,0.3,99,0.503333
1040,3,27,0.3,99,0.503333
680,2,27,0.3,99,0.503333
356,1,30,0.3,99,0.503333


Oddly enough - or maybe not - the songs being split into 3s to increase the amount of data 10 times did not change the results at all.